In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import sys
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision.models import ResNet50_Weights

# ==========================================
# 1. DATA SETUP (Download or Locate)
# ==========================================
print("--- Step 1: Setting up Dataset ---")

# Option A: Check if data is already attached (Standard Kaggle way)
if os.path.exists('../input/aptos2019-blindness-detection/train_images/'):
    print("✅ Found dataset in Kaggle Input directory.")
    ROOT_DIR = '../input/aptos2019-blindness-detection/train_images/'
    CSV_PATH = '../input/aptos2019-blindness-detection/train.csv'

# Option B: Download via API (Forces Overwrite to prevent freezing)
else:
    print("⚠️ Dataset not in input. Attempting download via API...")
    
    # 1. Set credentials 
    os.environ['KAGGLE_USERNAME'] = "username"
    os.environ['KAGGLE_KEY'] = "api_key"

    # 2. Download (force overwrite)
    !kaggle competitions download -c aptos2019-blindness-detection --force

    # 3. Unzip with -o (OVERWRITE) to prevent asking "replace?"
    print("Unzipping...")
    !unzip -qo aptos2019-blindness-detection.zip -d ./dataset
    
    # ROOT_DIR = './aptos_data/train_images/'
    # CSV_PATH = './aptos_data/train.csv'
    print("✅ Download and extraction complete.")

--- Step 1: Setting up Dataset ---
⚠️ Dataset not in input. Attempting download via API...
100%|███████████████████████████████████████| 9.51G/9.51G [00:55<00:00, 185MB/s]

Unzipping...
✅ Download and extraction complete.


In [3]:
!pip install torch torchvision timm scikit-learn tqdm

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
class APTOSDataset(Dataset):

    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        img_name = self.df.loc[idx, "id_code"]
        path = os.path.join(self.image_dir, img_name + ".png")

        try:
            image = Image.open(path).convert("RGB")
        except:
            image = Image.new("RGB", (300,300))

        label = self.df.loc[idx, "diagnosis"]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [7]:
def get_transforms(phase):

    if phase == "train":
        return transforms.Compose([
            transforms.Resize((300,300)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2,0.2,0.2,0.05),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],
                                 [0.229,0.224,0.225])
        ])

    else:
        return transforms.Compose([
            transforms.Resize((300,300)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],
                                 [0.229,0.224,0.225])
        ])

In [8]:
df = pd.read_csv("/kaggle/working/dataset/train.csv")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["diagnosis"],
    random_state=42
)

train_dataset = APTOSDataset(
    train_df,
    "/kaggle/working/dataset/train_images",
    get_transforms("train")
)

val_dataset = APTOSDataset(
    val_df,
    "/kaggle/working/dataset/train_images",
    get_transforms("val")
)

train_loader = DataLoader(train_dataset,
                          batch_size=16,
                          shuffle=True,
                          num_workers=2,
                          pin_memory=True)

val_loader = DataLoader(val_dataset,
                        batch_size=16,
                        shuffle=False,
                        num_workers=2,
                        pin_memory=True)

In [9]:
class DRModel(nn.Module):

    def __init__(self, num_classes=5):
        super().__init__()

        self.backbone = models.efficientnet_b3(weights="DEFAULT")

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [10]:
class FocalLoss(nn.Module):

    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()

In [11]:
model = DRModel().to(device)

criterion = FocalLoss(gamma=2)

optimizer = optim.Adam(model.parameters(), lr=2e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3
)

scaler = torch.cuda.amp.GradScaler()

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 177MB/s] 
/tmp/ipykernel_55/599141349.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [12]:
def train_epoch(model, loader):

    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    return total_loss/len(loader), acc

In [13]:
def validate_tta(model, loader):

    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)
            labels = labels.to(device)

            # TTA: original + flipped
            outputs1 = model(images)
            outputs2 = model(torch.flip(images, dims=[3]))

            outputs = (outputs1 + outputs2) / 2

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = outputs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    precision = precision_score(all_labels, all_preds, average="weighted")
    recall = recall_score(all_labels, all_preds, average="weighted")

    return total_loss/len(loader), acc, precision, recall, f1, all_labels, all_preds

In [14]:
EPOCHS = 25
best_f1 = 0

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_epoch(model, train_loader)

    val_loss, val_acc, precision, recall, f1, true, pred = validate_tta(model, val_loader)

    scheduler.step(val_loss)

    print(f"Train Acc: {train_acc:.4f}")
    print(f"Val Acc: {val_acc:.4f}")
    print(f"F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "/kaggle/working/best_efficientnet_model.pth")
        print("Model saved.")


Epoch 1/25


  0%|          | 0/184 [00:00<?, ?it/s]/tmp/ipykernel_55/1065738902.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 46/46 [00:45<00:00,  1.01it/s]


Train Acc: 0.7214
Val Acc: 0.7640
F1: 0.7474
Model saved.

Epoch 2/25


  0%|          | 0/184 [00:00<?, ?it/s]/tmp/ipykernel_55/1065738902.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 46/46 [00:45<00:00,  1.01it/s]


Train Acc: 0.7989
Val Acc: 0.8158
F1: 0.8154
Model saved.

Epoch 3/25


...

Train Acc: 0.9515
Val Acc: 0.8336
F1: 0.8289

Epoch 24/25


  0%|          | 0/184 [00:00<?, ?it/s]/tmp/ipykernel_55/1065738902.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 46/46 [00:45<00:00,  1.02it/s]


Train Acc: 0.9546
Val Acc: 0.8445
F1: 0.8370

Epoch 25/25


  0%|          | 0/184 [00:00<?, ?it/s]/tmp/ipykernel_55/1065738902.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 46/46 [00:44<00:00,  1.04it/s]

Train Acc: 0.9454
Val Acc: 0.8417
F1: 0.8340
